# ASG Airlines — End-to-End Data Engineering Pipeline

# Data Profiling & Quality Assessment

# Objective

Build a reliable end-to-end data engineering pipeline for ASG Airlines that ingests, validates, cleans transforms and models operational flight data for analytics and business intelligence.

# Pipeline Architecture

Source - Bronze - Data Quality - Silver - Gold - Power BI

# 1. Business Problem

ASG Airlines receives operational data from multiple systems including booking platforms, scheduling systems and airport logs.

The supplied dataset contains intentional data-quality issues such as:

- Missing values
- Duplicate records
- Corrupted or malformed identifiers
- Inconsistent time information
- Overnight flights
- Invalid operational records

These issues can result in inaccurate flight durations, route-level analysis, delay/anomaly identification and business reporting.

The objective of this project is to develop a reliable data pipeline that converts the raw operational data into validated, standardized and analytics-ready datasets.

Loading the Excel File.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = Path("../data/raw/UseCase - Airlines.xlsx")

excel_file = pd.ExcelFile(RAW_FILE)

excel_file.sheet_names

flights = pd.read_excel(RAW_FILE, sheet_name="flights")
bookings = pd.read_excel(RAW_FILE, sheet_name="bookings")
passengers = pd.read_excel(RAW_FILE, sheet_name="passengers")
payments = pd.read_excel(RAW_FILE, sheet_name="payments")
print("Flights:", flights.shape)
print("Bookings:", bookings.shape)
print("Passengers:", passengers.shape)
print("Payments:", payments.shape)

Flights: (1020, 7)
Bookings: (1000, 9)
Passengers: (1039, 9)
Payments: (1000, 4)


In [ ]:
def profile_dataframe(df, name):
    profile = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_count": df.isna().sum().values,
        "missing_percentage": (
            df.isna().mean().values * 100
        ).round(2),
        "unique_count": df.nunique().values
    })
    
    profile.insert(0, "dataset", name)
    
    return profile
flights_profile = profile_dataframe(flights, "flights")
bookings_profile = profile_dataframe(bookings, "bookings")
passengers_profile = profile_dataframe(passengers, "passengers")
payments_profile = profile_dataframe(payments, "payments")

flights_profile

,dataset,column,dtype,missing_count,missing_percentage,unique_count
0,flights,flight_id,object,0,0.00,1004
1,flights,airline,object,41,4.02,5
2,flights,source,object,0,0.00,6
3,flights,destination,object,0,0.00,6
4,flights,departure_time,datetime64[ns],0,0.00,981
5,flights,arrival_time,datetime64[ns],0,0.00,969
6,flights,duration,object,0,0.00,270


In [8]:
all_profiles = pd.concat(
    [
        flights_profile,
        bookings_profile,
        passengers_profile,
        payments_profile
    ],
    ignore_index=True
)

all_profiles

Path("../output").mkdir(exist_ok=True)

all_profiles.to_csv(
    "../output/data_profile_report.csv",
    index=False
)

# 7. Data Profiling

The raw datasets are profiled before any transformation is applied.

In [10]:
for name, df in {
    "Flights": flights,
    "Bookings": bookings,
    "Passengers": passengers,
    "Payments": payments
}.items():
    
    print(f"\n{'-'*50}")
    print(name)
    print(f"{'-'*50}")
    print(df.columns.tolist())


--------------------------------------------------
Flights
--------------------------------------------------
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']

--------------------------------------------------
Bookings
--------------------------------------------------
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone']

--------------------------------------------------
Passengers
--------------------------------------------------
['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']

--------------------------------------------------
Payments
--------------------------------------------------
['payment_id', 'booking_id', 'amount', 'payment_method']


In [11]:
display(flights.head())
display(bookings.head())
display(passengers.head())
display(payments.head())

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


In [12]:
for name, df in {
    "Flights": flights,
    "Bookings": bookings,
    "Passengers": passengers,
    "Payments": payments
}.items():
    
    print(f"\n{name}")
    print("-" * 50)
    print(df.dtypes)


Flights
--------------------------------------------------
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

Bookings
--------------------------------------------------
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object

Passengers
--------------------------------------------------
passenger_id             object
first_name               object
last_name                object
age                       int64
gender                   object
email              

In [13]:
def missing_report(df, dataset_name):
    
    result = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percentage": (
            df.isna().mean().values * 100
        ).round(2)
    })
    
    result.insert(0, "dataset", dataset_name)
    
    return result


for name, df in {
    "Flights": flights,
    "Bookings": bookings,
    "Passengers": passengers,
    "Payments": payments
}.items():
    
    print(f"\n{name}")
    display(missing_report(df, name))


Flights


,dataset,column,missing_count,missing_percentage
0,Flights,flight_id,0,0.00
1,Flights,airline,41,4.02
2,Flights,source,0,0.00
3,Flights,destination,0,0.00
4,Flights,departure_time,0,0.00
5,Flights,arrival_time,0,0.00
6,Flights,duration,0,0.00



Bookings


,dataset,column,missing_count,missing_percentage
0,Bookings,booking_id,0,0.0
1,Bookings,passenger_id,0,0.0
2,Bookings,flight_id,0,0.0
3,Bookings,booking_date,0,0.0
4,Bookings,status,45,4.5
5,Bookings,passport_number,0,0.0
6,Bookings,seat_number,0,0.0
7,Bookings,emergency_contact_name,0,0.0
8,Bookings,emergency_contact_phone,0,0.0



Passengers


,dataset,column,missing_count,missing_percentage
0,Passengers,passenger_id,0,0.00
1,Passengers,first_name,0,0.00
2,Passengers,last_name,10,0.96
3,Passengers,age,0,0.00
4,Passengers,gender,0,0.00
5,Passengers,email,0,0.00
6,Passengers,phone,0,0.00
7,Passengers,aadhaar_id,0,0.00
8,Passengers,date_of_birth,0,0.00



Payments


,dataset,column,missing_count,missing_percentage
0,Payments,payment_id,0,0.0
1,Payments,booking_id,0,0.0
2,Payments,amount,48,4.8
3,Payments,payment_method,0,0.0


In [14]:
for name, df in {
    "Flights": flights,
    "Bookings": bookings,
    "Passengers": passengers,
    "Payments": payments
}.items():
    
    print(f"{name}")
    print("Total rows:", len(df))
    print("Exact duplicate rows:", df.duplicated().sum())
    print()

Flights
Total rows: 1020
Exact duplicate rows: 15

Bookings
Total rows: 1000
Exact duplicate rows: 0

Passengers
Total rows: 1039
Exact duplicate rows: 0

Payments
Total rows: 1000
Exact duplicate rows: 0



In [15]:
flights["airline"].value_counts(dropna=False)
print("Sources:")
display(flights["source"].value_counts(dropna=False))

print("Destinations:")
display(flights["destination"].value_counts(dropna=False))

Sources:


source
BOM    207
HYD    180
CCU    174
DEL    163
MAA    157
BLR    139
Name: count, dtype: int64

Destinations:


destination
DEL    200
CCU    188
BOM    174
BLR    165
MAA    151
HYD    142
Name: count, dtype: int64

In [16]:
bookings["status"].value_counts(dropna=False)
payments["payment_method"].value_counts(dropna=False)
payments["amount"].describe()

count         952
unique        922
top       INVALID
freq           30
Name: amount, dtype: object

In [17]:
passenger_profile = pd.DataFrame({
    "column": passengers.columns,
    "dtype": passengers.dtypes.astype(str).values,
    "missing_count": passengers.isna().sum().values,
    "unique_count": passengers.nunique().values
})

display(passenger_profile)

,column,dtype,missing_count,unique_count
0,passenger_id,object,0,1000
1,first_name,object,0,31
2,last_name,object,10,38
3,age,int64,0,89
4,gender,object,0,2
5,email,object,0,1039
6,phone,object,0,1039
7,aadhaar_id,int64,0,1039
8,date_of_birth,datetime64[ns],0,1011


In [19]:
profiles = []

datasets = {
    "flights": flights,
    "bookings": bookings,
    "passengers": passengers,
    "payments": payments
}

for name, df in datasets.items():
    temp = profile_dataframe(df, name)
    profiles.append(temp)

full_profile = pd.concat(profiles, ignore_index=True)

display(full_profile)

full_profile.to_csv(
    "../output/raw_data_profile.csv",
    index=False
)

,dataset,column,dtype,missing_count,missing_percentage,unique_count
0,flights,flight_id,object,0,0.00,1004
1,flights,airline,object,41,4.02,5
2,flights,source,object,0,0.00,6
3,flights,destination,object,0,0.00,6
4,flights,departure_time,datetime64[ns],0,0.00,981
5,flights,arrival_time,datetime64[ns],0,0.00,969
6,flights,duration,object,0,0.00,270
7,bookings,booking_id,object,0,0.00,1000
8,bookings,passenger_id,object,0,0.00,636
9,bookings,flight_id,object,0,0.00,984


In [23]:
print("Duplicate payment_id:", payments["payment_id"].duplicated().sum())
print(payments[payments["payment_id"].duplicated(keep=False)].sort_values("payment_id").head(20))
print(payments["payment_method"].value_counts(dropna=False))
print(bookings["status"].value_counts(dropna=False))
print(passengers[["first_name", "last_name", "email", "phone", "aadhaar", "dob"]].isna().sum())
print(passengers["aadhaar"].astype(str).str.len().value_counts(dropna=False))
print(flights.isna().sum())
print(flights["airline"].value_counts(dropna=False))
print(flights["flight_id"].duplicated().sum())

Duplicate payment_id: 0
Empty DataFrame
Columns: [payment_id, booking_id, amount, payment_method]
Index: []
payment_method
UPI           358
CARD          329
NETBANKING    313
Name: count, dtype: int64
status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64


KeyError: "['aadhaar', 'dob'] not in index"

In [25]:
print("========== PASSENGER DATA PROFILING ==========\n")

# 1. Missing values
print("1. Missing Values:")
print(passengers.isna().sum())

# 2. Duplicate passenger IDs
print("\n2. Duplicate passenger_id:")
print(passengers["passenger_id"].duplicated().sum())

# 3. Aadhaar ID length
print("\n3. Aadhaar ID Length Distribution:")
print(
    passengers["aadhaar_id"]
    .astype("string")
    .str.len()
    .value_counts(dropna=False)
)

# 4. Gender distribution
print("\n4. Gender Distribution:")
print(passengers["gender"].value_counts(dropna=False))

# 5. Age statistics
print("\n5. Age Statistics:")
print(passengers["age"].describe())

# 6. Missing PII / personal fields
print("\n6. Missing Personal Information:")
print("Missing email:", passengers["email"].isna().sum())
print("Missing phone:", passengers["phone"].isna().sum())
print("Missing date_of_birth:", passengers["date_of_birth"].isna().sum())

print("\n========== PROFILING COMPLETE ==========")

========== PASSENGER DATA PROFILING ==========

1. Missing Values:
passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64

2. Duplicate passenger_id:
39

3. Aadhaar ID Length Distribution:
aadhaar_id
12    925
11    109
10      5
Name: count, dtype: Int64

4. Gender Distribution:
gender
M    545
F    494
Name: count, dtype: int64

5. Age Statistics:
count    1039.000000
mean       43.304139
std        25.900616
min         1.000000
25%        20.000000
50%        43.000000
75%        64.000000
max        89.000000
Name: age, dtype: float64

6. Missing Personal Information:
Missing email: 0
Missing phone: 0
Missing date_of_birth: 0

========== PROFILING COMPLETE ==========


In [26]:

print("========== FLIGHTS DATA PROFILING ==========\n")

# 1. Dataset shape
print("1. Dataset Shape:")
print(flights.shape)

# 2. Column names
print("\n2. Columns:")
print(flights.columns.tolist())

# 3. Data types
print("\n3. Data Types:")
print(flights.dtypes)

# 4. Missing values
print("\n4. Missing Values:")
print(flights.isna().sum())

# 5. Exact duplicate rows
print("\n5. Exact Duplicate Rows:")
print(flights.duplicated().sum())

# 6. Duplicate flight IDs
print("\n6. Duplicate flight_id:")
print(flights["flight_id"].duplicated().sum())

# 7. Flight ID examples
print("\n7. Sample Flight IDs:")
print(flights["flight_id"].head(20).to_string(index=False))

# 8. Airline distribution
print("\n8. Airline Distribution:")
print(flights["airline"].value_counts(dropna=False))

# 9. Source distribution
print("\n9. Source Distribution:")
print(flights["source"].value_counts(dropna=False).head(20))

# 10. Destination distribution
print("\n10. Destination Distribution:")
print(flights["destination"].value_counts(dropna=False).head(20))

print("\n========== PROFILING COMPLETE ==========")

========== FLIGHTS DATA PROFILING ==========

1. Dataset Shape:
(1020, 7)

2. Columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']

3. Data Types:
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

4. Missing Values:
flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

5. Exact Duplicate Rows:
15

6. Duplicate flight_id:
16

7. Sample Flight IDs:
SJ010
AI155
UK094
AI245
AI192
SJ158
6F196
AI080
6F025
6F251
AI137
6F026
UK167
UK027
AI069
AI058
AI140
SJ239
6F099
AI048

8. Airline Distribution:
airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           41
UNKNOWN       31
Name: count, dtype: int64

9. Source Dis

In [27]:
# ============================================
# FLIGHT TIME & DURATION PROFILING
# ============================================

print("========== FLIGHT TIME & DURATION PROFILING ==========\n")

# 1. Check datetime nulls
print("1. Missing Departure Times:")
print(flights["departure_time"].isna().sum())

print("\nMissing Arrival Times:")
print(flights["arrival_time"].isna().sum())

# 2. Check whether arrival is before departure
arrival_before_departure = flights[
    flights["arrival_time"] < flights["departure_time"]
]

print("\n2. Arrival Before Departure:")
print("Count:", len(arrival_before_departure))

if len(arrival_before_departure) > 0:
    print(arrival_before_departure[
        ["flight_id", "departure_time", "arrival_time", "duration"]
    ].to_string(index=False))

# 3. Cross-day / overnight flights
overnight_flights = flights[
    flights["arrival_time"].dt.date > flights["departure_time"].dt.date
]

print("\n3. Cross-Day / Overnight Flights:")
print("Count:", len(overnight_flights))

if len(overnight_flights) > 0:
    print(overnight_flights[
        ["flight_id", "departure_time", "arrival_time", "duration"]
    ].head(20).to_string(index=False))

# 4. Duration data types
print("\n4. Duration Data Types:")
print(flights["duration"].apply(type).value_counts())

# 5. Duration examples
print("\n5. Sample Duration Values:")
print(flights["duration"].value_counts(dropna=False).head(20))

# 6. Duration missing values
print("\n6. Missing Duration:")
print(flights["duration"].isna().sum())

print("\n========== PROFILING COMPLETE ==========")

========== FLIGHT TIME & DURATION PROFILING ==========

1. Missing Departure Times:
0

Missing Arrival Times:
0

2. Arrival Before Departure:
Count: 1
flight_id      departure_time        arrival_time            duration
    SJ192 2026-04-19 18:45:42 2026-04-18 23:45:42 1899-12-29 05:00:00

3. Cross-Day / Overnight Flights:
Count: 124
flight_id          departure_time            arrival_time duration
    SJ010 2026-04-20 23:38:41.701 2026-04-21 02:32:41.701 02:54:00
    AI155 2026-04-20 23:35:41.703 2026-04-21 01:23:41.703 01:48:00
    UK094 2026-04-20 23:26:41.702 2026-04-21 01:11:41.702 01:45:00
    AI245 2026-04-20 23:07:41.704 2026-04-21 01:43:41.704 02:36:00
    AI192 2026-04-20 23:05:41.703 2026-04-21 04:04:41.703 04:59:00
    SJ158 2026-04-20 23:05:41.703 2026-04-21 01:24:41.703 02:19:00
    6F196 2026-04-20 23:04:41.703 2026-04-21 00:47:41.703 01:43:00
    AI080 2026-04-20 23:03:41.702 2026-04-21 00:35:41.702 01:32:00
    AI137 2026-04-20 22:49:41.702 2026-04-21 01:17:41.702 02

In [28]:
# ============================================
# FLIGHT DURATION CONSISTENCY CHECK
# ============================================

print("========== DURATION CONSISTENCY CHECK ==========\n")

# Calculate duration directly from timestamps
calculated_duration = (
    flights["arrival_time"] - flights["departure_time"]
).dt.total_seconds() / 60

# Create a temporary comparison DataFrame
duration_check = flights[
    ["flight_id", "departure_time", "arrival_time", "duration"]
].copy()

duration_check["calculated_duration_minutes"] = calculated_duration

# Convert stored duration to minutes
def duration_to_minutes(value):
    if hasattr(value, "hour") and hasattr(value, "minute"):
        return (
            value.hour * 60
            + value.minute
            + value.second / 60
            + value.microsecond / 60000000
        )
    return None

duration_check["stored_duration_minutes"] = (
    duration_check["duration"].apply(duration_to_minutes)
)

# Difference between stored and calculated duration
duration_check["difference_minutes"] = (
    duration_check["stored_duration_minutes"]
    - duration_check["calculated_duration_minutes"]
)

# Records where duration doesn't match
duration_mismatch = duration_check[
    duration_check["difference_minutes"].abs() > 1
]

print("Total flights:", len(duration_check))
print("Duration mismatches:", len(duration_mismatch))

print("\nSample mismatches:")
print(
    duration_mismatch[
        [
            "flight_id",
            "departure_time",
            "arrival_time",
            "duration",
            "stored_duration_minutes",
            "calculated_duration_minutes",
            "difference_minutes"
        ]
    ].head(20).to_string(index=False)
)

print("\n========== CHECK COMPLETE ==========")

========== DURATION CONSISTENCY CHECK ==========

Total flights: 1020
Duration mismatches: 1

Sample mismatches:
flight_id      departure_time        arrival_time            duration  stored_duration_minutes  calculated_duration_minutes  difference_minutes
    SJ192 2026-04-19 18:45:42 2026-04-18 23:45:42 1899-12-29 05:00:00                    300.0                      -1140.0              1440.0

========== CHECK COMPLETE ==========


In [29]:
# ============================================
# STEP 3A — ID & REFERENTIAL INTEGRITY CHECK
# ============================================

print("========== ID & REFERENTIAL INTEGRITY ==========\n")

# ------------------------------------------------
# 1. Show ID columns
# ------------------------------------------------
print("1. ID Columns:")
print("Flights   :", flights["flight_id"].head(10).tolist())
print("Bookings  :", bookings["booking_id"].head(10).tolist())
print("Passengers:", passengers["passenger_id"].head(10).tolist())
print("Payments  :", payments["payment_id"].head(10).tolist())

# ------------------------------------------------
# 2. Duplicate IDs
# ------------------------------------------------
print("\n2. Duplicate ID Counts:")
print("flight_id    :", flights["flight_id"].duplicated().sum())
print("booking_id   :", bookings["booking_id"].duplicated().sum())
print("passenger_id :", passengers["passenger_id"].duplicated().sum())
print("payment_id   :", payments["payment_id"].duplicated().sum())

# ------------------------------------------------
# 3. Booking → Flight relationship
# ------------------------------------------------
missing_flights = bookings[
    ~bookings["flight_id"].isin(flights["flight_id"])
]

print("\n3. Booking → Flight Integrity:")
print("Bookings with missing flight_id reference:", len(missing_flights))

if len(missing_flights) > 0:
    print(missing_flights[
        ["booking_id", "flight_id"]
    ].head(20).to_string(index=False))

# ------------------------------------------------
# 4. Booking → Passenger relationship
# ------------------------------------------------
missing_passengers = bookings[
    ~bookings["passenger_id"].isin(passengers["passenger_id"])
]

print("\n4. Booking → Passenger Integrity:")
print("Bookings with missing passenger_id reference:", len(missing_passengers))

if len(missing_passengers) > 0:
    print(missing_passengers[
        ["booking_id", "passenger_id"]
    ].head(20).to_string(index=False))

# ------------------------------------------------
# 5. Payment → Booking relationship
# ------------------------------------------------
missing_bookings = payments[
    ~payments["booking_id"].isin(bookings["booking_id"])
]

print("\n5. Payment → Booking Integrity:")
print("Payments with missing booking_id reference:", len(missing_bookings))

if len(missing_bookings) > 0:
    print(missing_bookings[
        ["payment_id", "booking_id"]
    ].head(20).to_string(index=False))

# ------------------------------------------------
# 6. Null IDs
# ------------------------------------------------
print("\n6. Missing IDs:")
print("Flights flight_id:", flights["flight_id"].isna().sum())
print("Bookings booking_id:", bookings["booking_id"].isna().sum())
print("Bookings flight_id:", bookings["flight_id"].isna().sum())
print("Bookings passenger_id:", bookings["passenger_id"].isna().sum())
print("Passengers passenger_id:", passengers["passenger_id"].isna().sum())
print("Payments payment_id:", payments["payment_id"].isna().sum())
print("Payments booking_id:", payments["booking_id"].isna().sum())

print("\n========== CHECK COMPLETE ==========")

========== ID & REFERENTIAL INTEGRITY ==========

1. ID Columns:
Flights   : ['SJ010', 'AI155', 'UK094', 'AI245', 'AI192', 'SJ158', '6F196', 'AI080', '6F025', '6F251']
Bookings  : ['B1000', 'B1001', 'B1002', 'B1003', 'B1004', 'B1005', 'B1006', 'B1007', 'B1008', 'B1009']
Passengers: ['P1000', 'P1001', 'P1002', 'P1003', 'P1004', 'P1005', 'P1006', 'P1007', 'P1008', 'P1009']
Payments  : ['PAY1000', 'PAY1001', 'PAY1002', 'PAY1003', 'PAY1004', 'PAY1005', 'PAY1006', 'PAY1007', 'PAY1008', 'PAY1009']

2. Duplicate ID Counts:
flight_id    : 16
booking_id   : 0
passenger_id : 39
payment_id   : 0

3. Booking → Flight Integrity:
Bookings with missing flight_id reference: 0

4. Booking → Passenger Integrity:
Bookings with missing passenger_id reference: 0

5. Payment → Booking Integrity:
Payments with missing booking_id reference: 0

6. Missing IDs:
Flights flight_id: 0
Bookings booking_id: 0
Bookings flight_id: 0
Bookings passenger_id: 0
Passengers passenger_id: 0
Payments payment_id: 0
Payments bo

In [30]:
# ============================================
# STEP 3B — DUPLICATE ID INVESTIGATION
# ============================================

print("========== DUPLICATE ID INVESTIGATION ==========\n")

# --------------------------------------------
# 1. Duplicate Flight IDs
# --------------------------------------------
duplicate_flights = flights[
    flights["flight_id"].duplicated(keep=False)
].sort_values("flight_id")

print("1. Duplicate Flight IDs")
print("Number of affected rows:", len(duplicate_flights))
print("Number of unique duplicated IDs:",
      duplicate_flights["flight_id"].nunique())

print("\nDuplicate flight records:")
print(
    duplicate_flights[
        [
            "flight_id",
            "airline",
            "source",
            "destination",
            "departure_time",
            "arrival_time",
            "duration"
        ]
    ].to_string(index=False)
)

# --------------------------------------------
# 2. Duplicate Passenger IDs
# --------------------------------------------
duplicate_passengers = passengers[
    passengers["passenger_id"].duplicated(keep=False)
].sort_values("passenger_id")

print("\n\n2. Duplicate Passenger IDs")
print("Number of affected rows:", len(duplicate_passengers))
print("Number of unique duplicated IDs:",
      duplicate_passengers["passenger_id"].nunique())

# DO NOT print Aadhaar, phone or email
print("\nDuplicate passenger records:")
print(
    duplicate_passengers[
        [
            "passenger_id",
            "first_name",
            "last_name",
            "age",
            "gender"
        ]
    ].to_string(index=False)
)

print("\n========== INVESTIGATION COMPLETE ==========")

========== DUPLICATE ID INVESTIGATION ==========

1. Duplicate Flight IDs
Number of affected rows: 32
Number of unique duplicated IDs: 16

Duplicate flight records:
flight_id   airline source destination          departure_time            arrival_time duration
    6F250   UNKNOWN    DEL         BLR 2026-04-20 03:26:41.701 2026-04-20 07:30:41.701 04:04:00
    6F250   UNKNOWN    CCU         BLR 2026-04-20 02:23:41.702 2026-04-20 02:56:41.702 00:33:00
    AI020       NaN    BOM         BLR 2026-04-19 03:53:41.701 2026-04-19 06:21:41.701 02:28:00
    AI020       NaN    BOM         BLR 2026-04-19 03:53:41.701 2026-04-19 06:21:41.701 02:28:00
    AI031 Air India    DEL         MAA 2026-04-20 13:05:41.701 2026-04-20 16:11:41.701 03:06:00
    AI031 Air India    DEL         MAA 2026-04-20 13:05:41.701 2026-04-20 16:11:41.701 03:06:00
    AI043 Air India    CCU         DEL 2026-04-19 00:28:41.701 2026-04-19 04:37:41.701 04:09:00
    AI043 Air India    CCU         DEL 2026-04-19 00:28:41.701 2026

In [31]:
# ============================================
# STEP 3C — DATA QUALITY RULEBOOK
# ============================================

dq_rules = [

    # ---------------- FLIGHTS ----------------
    {
        "rule_id": "F001",
        "dataset": "flights",
        "column": "flight_id",
        "rule": "Flight ID must not be null",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "F002",
        "dataset": "flights",
        "column": "flight_id",
        "rule": "Duplicate flight IDs must be investigated; do not automatically delete",
        "severity": "HIGH",
        "action": "FLAG"
    },
    {
        "rule_id": "F003",
        "dataset": "flights",
        "column": "airline",
        "rule": "Airline should not be null",
        "severity": "MEDIUM",
        "action": "FLAG"
    },
    {
        "rule_id": "F004",
        "dataset": "flights",
        "column": "airline",
        "rule": "UNKNOWN airline should be flagged",
        "severity": "MEDIUM",
        "action": "FLAG"
    },
    {
        "rule_id": "F005",
        "dataset": "flights",
        "column": "record",
        "rule": "Exact duplicate rows should be removed",
        "severity": "HIGH",
        "action": "DEDUPLICATE"
    },
    {
        "rule_id": "F006",
        "dataset": "flights",
        "column": "departure_time",
        "rule": "Departure timestamp must be valid",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "F007",
        "dataset": "flights",
        "column": "arrival_time",
        "rule": "Arrival timestamp must be valid",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "F008",
        "dataset": "flights",
        "column": "arrival_time",
        "rule": "Arrival must not occur before departure",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "F009",
        "dataset": "flights",
        "column": "duration",
        "rule": "Duration should be derived from departure and arrival timestamps",
        "severity": "HIGH",
        "action": "RECALCULATE"
    },
    {
        "rule_id": "F010",
        "dataset": "flights",
        "column": "arrival_time",
        "rule": "Cross-day flights must be identified as overnight flights",
        "severity": "INFO",
        "action": "FLAG"
    },

    # ---------------- BOOKINGS ----------------
    {
        "rule_id": "B001",
        "dataset": "bookings",
        "column": "booking_id",
        "rule": "Booking ID must be unique",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "B002",
        "dataset": "bookings",
        "column": "status",
        "rule": "Status must be CONFIRMED, CANCELLED or PENDING",
        "severity": "HIGH",
        "action": "FLAG"
    },
    {
        "rule_id": "B003",
        "dataset": "bookings",
        "column": "flight_id",
        "rule": "Flight reference must exist in flights",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "B004",
        "dataset": "bookings",
        "column": "passenger_id",
        "rule": "Passenger reference must exist in passengers",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },

    # ---------------- PASSENGERS ----------------
    {
        "rule_id": "P001",
        "dataset": "passengers",
        "column": "passenger_id",
        "rule": "Passenger ID duplicates must be investigated",
        "severity": "HIGH",
        "action": "FLAG"
    },
    {
        "rule_id": "P002",
        "dataset": "passengers",
        "column": "last_name",
        "rule": "Last name should not be null",
        "severity": "MEDIUM",
        "action": "FLAG"
    },
    {
        "rule_id": "P003",
        "dataset": "passengers",
        "column": "aadhaar_id",
        "rule": "Aadhaar ID must contain the expected number of digits",
        "severity": "HIGH",
        "action": "FLAG"
    },
    {
        "rule_id": "P004",
        "dataset": "passengers",
        "column": "aadhaar_id",
        "rule": "Aadhaar ID must not be exposed in analytical outputs",
        "severity": "CRITICAL",
        "action": "MASK"
    },

    # ---------------- PAYMENTS ----------------
    {
        "rule_id": "PY001",
        "dataset": "payments",
        "column": "payment_id",
        "rule": "Payment ID must be unique",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "PY002",
        "dataset": "payments",
        "column": "booking_id",
        "rule": "Booking reference must exist in bookings",
        "severity": "CRITICAL",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "PY003",
        "dataset": "payments",
        "column": "amount",
        "rule": "Payment amount must be numeric",
        "severity": "HIGH",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "PY004",
        "dataset": "payments",
        "column": "amount",
        "rule": "Payment amount must not be null",
        "severity": "HIGH",
        "action": "QUARANTINE"
    },
    {
        "rule_id": "PY005",
        "dataset": "payments",
        "column": "payment_method",
        "rule": "Payment method must be UPI, CARD or NETBANKING",
        "severity": "MEDIUM",
        "action": "FLAG"
    }
]

print("Total DQ Rules:", len(dq_rules))

for rule in dq_rules:
    print(
        rule["rule_id"],
        "|",
        rule["dataset"],
        "|",
        rule["severity"],
        "|",
        rule["action"]
    )

Total DQ Rules: 23
F001 | flights | CRITICAL | QUARANTINE
F002 | flights | HIGH | FLAG
F003 | flights | MEDIUM | FLAG
F004 | flights | MEDIUM | FLAG
F005 | flights | HIGH | DEDUPLICATE
F006 | flights | CRITICAL | QUARANTINE
F007 | flights | CRITICAL | QUARANTINE
F008 | flights | CRITICAL | QUARANTINE
F009 | flights | HIGH | RECALCULATE
F010 | flights | INFO | FLAG
B001 | bookings | CRITICAL | QUARANTINE
B002 | bookings | HIGH | FLAG
B003 | bookings | CRITICAL | QUARANTINE
B004 | bookings | CRITICAL | QUARANTINE
P001 | passengers | HIGH | FLAG
P002 | passengers | MEDIUM | FLAG
P003 | passengers | HIGH | FLAG
P004 | passengers | CRITICAL | MASK
PY001 | payments | CRITICAL | QUARANTINE
PY002 | payments | CRITICAL | QUARANTINE
PY003 | payments | HIGH | QUARANTINE
PY004 | payments | HIGH | QUARANTINE
PY005 | payments | MEDIUM | FLAG


In [33]:
from pathlib import Path

# Project root = one level above the notebook directory
project_root = Path("..")

folders = [
    "data/raw",
    "data/bronze",
    "data/silver",
    "data/gold",
    "data/quarantine",
    "notebooks",
    "src",
    "tests",
    "output",
    "powerbi",
    "docs"
]

for folder in folders:
    (project_root / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")

for folder in folders:
    print(" ", folder)

Project folders created successfully.
  data/raw
  data/bronze
  data/silver
  data/gold
  data/quarantine
  notebooks
  src
  tests
  output
  powerbi
  docs


In [34]:
# ============================================
# STEP 4 — BRONZE LAYER
# ============================================

bronze_path = project_root / "data" / "bronze"

# Save raw datasets without cleaning
flights.to_csv(bronze_path / "flights.csv", index=False)
bookings.to_csv(bronze_path / "bookings.csv", index=False)
passengers.to_csv(bronze_path / "passengers.csv", index=False)
payments.to_csv(bronze_path / "payments.csv", index=False)

print("========== BRONZE INGESTION COMPLETE ==========")

print("Flights   :", len(flights), "records")
print("Bookings  :", len(bookings), "records")
print("Passengers:", len(passengers), "records")
print("Payments  :", len(payments), "records")

print("\nBronze location:", bronze_path.resolve())

========== BRONZE INGESTION COMPLETE ==========
Flights   : 1020 records
Bookings  : 1000 records
Passengers: 1039 records
Payments  : 1000 records

Bronze location: C:\Users\User\OneDrive\Desktop\Airlines_DataEngineering\data\bronze


In [48]:
# ============================================================
# STEP 5 — COMPLETE DATA QUALITY ENGINE
# ASG AIRLINES
# ============================================================

import pandas as pd
from datetime import datetime
from pathlib import Path

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

dq_output_path = project_root / "output"
quarantine_path = project_root / "data" / "quarantine"

dq_output_path.mkdir(parents=True, exist_ok=True)
quarantine_path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. STORAGE FOR DQ RESULTS
# ------------------------------------------------------------

dq_issues = []
quarantine_records = {
    "flights": [],
    "bookings": [],
    "passengers": [],
    "payments": []
}


def record_issue(
    dataset,
    rule_id,
    severity,
    action,
    failed_df,
    description
):
    """
    Record a DQ rule result.
    """

    count = len(failed_df)

    dq_issues.append({
        "dataset": dataset,
        "rule_id": rule_id,
        "severity": severity,
        "action": action,
        "failed_record_count": count,
        "description": description,
        "checked_at": datetime.now()
    })


def add_quarantine(
    dataset,
    rule_id,
    severity,
    failed_df,
    reason
):
    """
    Add failed records to the quarantine collection.
    """

    if len(failed_df) == 0:
        return

    temp = failed_df.copy()

    temp["dq_rule"] = rule_id
    temp["dq_severity"] = severity
    temp["dq_reason"] = reason

    quarantine_records[dataset].append(temp)


# ============================================================
# 3. FLIGHTS — F001 TO F010
# ============================================================

print("========== FLIGHTS DQ ==========")


# F001 — flight_id not null
failed = flights[flights["flight_id"].isna()]

record_issue(
    "flights",
    "F001",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Flight ID is missing"
)

add_quarantine(
    "flights",
    "F001",
    "CRITICAL",
    failed,
    "Missing flight_id"
)

print("F001:", len(failed))


# F002 — duplicate flight IDs
failed = flights[
    flights["flight_id"].duplicated(keep=False)
]

record_issue(
    "flights",
    "F002",
    "HIGH",
    "FLAG",
    failed,
    "Duplicate flight IDs detected"
)

print("F002:", len(failed))


# F003 — missing airline
failed = flights[
    flights["airline"].isna()
]

record_issue(
    "flights",
    "F003",
    "MEDIUM",
    "FLAG",
    failed,
    "Airline value is missing"
)

print("F003:", len(failed))


# F004 — UNKNOWN airline
failed = flights[
    flights["airline"]
    .astype("string")
    .str.upper()
    .eq("UNKNOWN")
]

record_issue(
    "flights",
    "F004",
    "MEDIUM",
    "FLAG",
    failed,
    "Airline is marked as UNKNOWN"
)

print("F004:", len(failed))


# F005 — exact duplicate rows
failed = flights[
    flights.duplicated(keep=False)
]

record_issue(
    "flights",
    "F005",
    "HIGH",
    "DEDUPLICATE",
    failed,
    "Exact duplicate flight records"
)

print("F005:", len(failed))


# F006 — invalid departure timestamp
departure_parsed = pd.to_datetime(
    flights["departure_time"],
    errors="coerce"
)

failed = flights[
    departure_parsed.isna()
]

record_issue(
    "flights",
    "F006",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Invalid departure timestamp"
)

add_quarantine(
    "flights",
    "F006",
    "CRITICAL",
    failed,
    "Invalid departure timestamp"
)

print("F006:", len(failed))


# F007 — invalid arrival timestamp
arrival_parsed = pd.to_datetime(
    flights["arrival_time"],
    errors="coerce"
)

failed = flights[
    arrival_parsed.isna()
]

record_issue(
    "flights",
    "F007",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Invalid arrival timestamp"
)

add_quarantine(
    "flights",
    "F007",
    "CRITICAL",
    failed,
    "Invalid arrival timestamp"
)

print("F007:", len(failed))


# F008 — arrival before departure
failed = flights[
    arrival_parsed < departure_parsed
]

record_issue(
    "flights",
    "F008",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Arrival occurs before departure"
)

add_quarantine(
    "flights",
    "F008",
    "CRITICAL",
    failed,
    "Arrival timestamp occurs before departure timestamp"
)

print("F008:", len(failed))


# F009 — duration recalculation
calculated_duration = (
    arrival_parsed - departure_parsed
).dt.total_seconds() / 60


def convert_duration_to_minutes(value):

    if pd.isna(value):
        return None

    if hasattr(value, "hour"):
        return (
            value.hour * 60
            + value.minute
            + value.second / 60
        )

    return None


stored_duration = flights["duration"].apply(
    convert_duration_to_minutes
)

failed = flights[
    stored_duration.notna()
    & calculated_duration.notna()
    & (
        stored_duration.round(2)
        != calculated_duration.round(2)
    )
]

record_issue(
    "flights",
    "F009",
    "HIGH",
    "RECALCULATE",
    failed,
    "Stored duration differs from timestamp-derived duration"
)

print("F009:", len(failed))


# F010 — overnight flight
overnight_mask = (
    arrival_parsed.dt.date
    > departure_parsed.dt.date
)

failed = flights[overnight_mask]

record_issue(
    "flights",
    "F010",
    "INFO",
    "FLAG",
    failed,
    "Flight crosses calendar day"
)

print("F010:", len(failed))


# ============================================================
# 4. BOOKINGS — B001 TO B004
# ============================================================

print("\n========== BOOKINGS DQ ==========")


# B001 — duplicate booking IDs
failed = bookings[
    bookings["booking_id"].duplicated(keep=False)
]

record_issue(
    "bookings",
    "B001",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Duplicate booking ID"
)

add_quarantine(
    "bookings",
    "B001",
    "CRITICAL",
    failed,
    "Duplicate booking_id"
)

print("B001:", len(failed))


# B002 — invalid booking status
valid_statuses = {
    "CONFIRMED",
    "CANCELLED",
    "PENDING"
}

failed = bookings[
    ~bookings["status"]
    .astype("string")
    .str.upper()
    .isin(valid_statuses)
]

record_issue(
    "bookings",
    "B002",
    "HIGH",
    "FLAG",
    failed,
    "Booking status is missing or invalid"
)

print("B002:", len(failed))


# B003 — flight reference must exist
failed = bookings[
    ~bookings["flight_id"].isin(
        flights["flight_id"]
    )
]

record_issue(
    "bookings",
    "B003",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Booking references a non-existent flight"
)

add_quarantine(
    "bookings",
    "B003",
    "CRITICAL",
    failed,
    "Invalid flight_id reference"
)

print("B003:", len(failed))


# B004 — passenger reference must exist
failed = bookings[
    ~bookings["passenger_id"].isin(
        passengers["passenger_id"]
    )
]

record_issue(
    "bookings",
    "B004",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Booking references a non-existent passenger"
)

add_quarantine(
    "bookings",
    "B004",
    "CRITICAL",
    failed,
    "Invalid passenger_id reference"
)

print("B004:", len(failed))


# ============================================================
# 5. PASSENGERS — P001 TO P004
# ============================================================

print("\n========== PASSENGERS DQ ==========")


# P001 — duplicate passenger IDs
failed = passengers[
    passengers["passenger_id"].duplicated(keep=False)
]

record_issue(
    "passengers",
    "P001",
    "HIGH",
    "FLAG",
    failed,
    "Duplicate passenger IDs detected"
)

print("P001:", len(failed))


# P002 — missing last name
failed = passengers[
    passengers["last_name"].isna()
]

record_issue(
    "passengers",
    "P002",
    "MEDIUM",
    "FLAG",
    failed,
    "Passenger last name is missing"
)

print("P002:", len(failed))


# P003 — invalid Aadhaar length
aadhaar_clean = (
    passengers["aadhaar_id"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
)

failed = passengers[
    aadhaar_clean.str.len() != 12
]

record_issue(
    "passengers",
    "P003",
    "HIGH",
    "FLAG",
    failed,
    "Aadhaar identifier does not contain 12 digits"
)

print("P003:", len(failed))


# P004 — PII protection
# We don't expose the actual Aadhaar values.

record_issue(
    "passengers",
    "P004",
    "CRITICAL",
    "MASK",
    passengers,
    "Passenger PII must be masked or protected in analytical outputs"
)

print("P004: PII protection rule applied")


# ============================================================
# 6. PAYMENTS — PY001 TO PY005
# ============================================================

print("\n========== PAYMENTS DQ ==========")


# PY001 — duplicate payment IDs
failed = payments[
    payments["payment_id"].duplicated(keep=False)
]

record_issue(
    "payments",
    "PY001",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Duplicate payment ID"
)

add_quarantine(
    "payments",
    "PY001",
    "CRITICAL",
    failed,
    "Duplicate payment_id"
)

print("PY001:", len(failed))


# PY002 — booking reference must exist
failed = payments[
    ~payments["booking_id"].isin(
        bookings["booking_id"]
    )
]

record_issue(
    "payments",
    "PY002",
    "CRITICAL",
    "QUARANTINE",
    failed,
    "Payment references a non-existent booking"
)

add_quarantine(
    "payments",
    "PY002",
    "CRITICAL",
    failed,
    "Invalid booking_id reference"
)

print("PY002:", len(failed))


# PY003 — amount must be numeric
numeric_amount = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

failed = payments[
    payments["amount"].notna()
    & numeric_amount.isna()
]

record_issue(
    "payments",
    "PY003",
    "HIGH",
    "QUARANTINE",
    failed,
    "Payment amount is not numeric"
)

add_quarantine(
    "payments",
    "PY003",
    "HIGH",
    failed,
    "Non-numeric payment amount"
)

print("PY003:", len(failed))


# PY004 — amount cannot be null
failed = payments[
    payments["amount"].isna()
]

record_issue(
    "payments",
    "PY004",
    "HIGH",
    "QUARANTINE",
    failed,
    "Payment amount is missing"
)

add_quarantine(
    "payments",
    "PY004",
    "HIGH",
    failed,
    "Missing payment amount"
)

print("PY004:", len(failed))


# PY005 — valid payment method
valid_methods = {
    "UPI",
    "CARD",
    "NETBANKING"
}

failed = payments[
    ~payments["payment_method"]
    .astype("string")
    .str.upper()
    .isin(valid_methods)
]

record_issue(
    "payments",
    "PY005",
    "MEDIUM",
    "FLAG",
    failed,
    "Payment method is invalid or missing"
)

print("PY005:", len(failed))


# ============================================================
# 7. CREATE DQ REPORT
# ============================================================

dq_report = pd.DataFrame(dq_issues)

dq_report = dq_report.sort_values(
    ["dataset", "rule_id"]
).reset_index(drop=True)

dq_report.to_csv(
    dq_output_path / "dq_report.csv",
    index=False
)


# ============================================================
# 8. CREATE QUARANTINE FILES
# ============================================================

for dataset, frames in quarantine_records.items():

    if frames:

        combined = pd.concat(
            frames,
            ignore_index=True
        )

        # Remove duplicate quarantine entries
        combined = combined.drop_duplicates()

        combined.to_csv(
            quarantine_path / f"{dataset}.csv",
            index=False
        )

        print(
            f"{dataset}: "
            f"{len(combined)} quarantined records"
        )


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("          ASG AIRLINES DQ SUMMARY")
print("=" * 60)

print("Total DQ Rules:", len(dq_report))
print(
    "Rules with violations:",
    (dq_report["failed_record_count"] > 0).sum()
)

print(
    "Total violation events:",
    dq_report["failed_record_count"].sum()
)

print("\nBy Dataset:")

dataset_summary = (
    dq_report
    .groupby("dataset")
    .agg(
        rules_checked=("rule_id", "count"),
        rules_with_issues=(
            "failed_record_count",
            lambda x: (x > 0).sum()
        ),
        violation_events=(
            "failed_record_count",
            "sum"
        )
    )
    .reset_index()
)

print(dataset_summary.to_string(index=False))

print("\nDQ report saved to:")
print(dq_output_path / "dq_report.csv")

print("\nQuarantine files saved to:")
print(quarantine_path)

========== FLIGHTS DQ ==========
F001: 0
F002: 32
F003: 41
F004: 31
F005: 30
F006: 0
F007: 0
F008: 1
F009: 1
F010: 124

========== BOOKINGS DQ ==========
B001: 0
B002: 75
B003: 0
B004: 0

========== PASSENGERS DQ ==========
P001: 75
P002: 10
P003: 114
P004: PII protection rule applied

========== PAYMENTS DQ ==========
PY001: 0
PY002: 0
PY003: 30
PY004: 48
PY005: 0
flights: 1 quarantined records
payments: 78 quarantined records


          ASG AIRLINES DQ SUMMARY
Total DQ Rules: 23
Rules with violations: 14
Total violation events: 1651

By Dataset:
   dataset  rules_checked  rules_with_issues  violation_events
  bookings              4                  1                75
   flights             10                  7               260
passengers              4                  4              1238
  payments              5                  2                78

DQ report saved to:
..\output\dq_report.csv

Quarantine files saved to:
..\data\quarantine


In [49]:
silver_path = project_root / "data" / "silver"
silver_path.mkdir(parents=True, exist_ok=True)

# Start from Bronze/raw data
silver_flights = flights.copy()

print("========== SILVER FLIGHTS ==========")
print("Initial records:", len(silver_flights))


# ------------------------------------------------------------
# 1. Standardize text columns
# ------------------------------------------------------------

text_columns = [
    "flight_id",
    "airline",
    "source",
    "destination"
]

for col in text_columns:
    silver_flights[col] = (
        silver_flights[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )


# ------------------------------------------------------------
# 2. Handle missing airline
# ------------------------------------------------------------

silver_flights["airline"] = (
    silver_flights["airline"]
    .fillna("UNKNOWN")
)


# ------------------------------------------------------------
# 3. Remove exact duplicate records
# ------------------------------------------------------------

before_dedup = len(silver_flights)

silver_flights = silver_flights.drop_duplicates()

removed_duplicates = (
    before_dedup - len(silver_flights)
)


# ------------------------------------------------------------
# 4. Convert timestamps
# ------------------------------------------------------------

silver_flights["departure_time"] = pd.to_datetime(
    silver_flights["departure_time"],
    errors="coerce"
)

silver_flights["arrival_time"] = pd.to_datetime(
    silver_flights["arrival_time"],
    errors="coerce"
)


# ------------------------------------------------------------
# 5. Calculate trusted duration
# ------------------------------------------------------------

silver_flights["duration_minutes"] = (
    silver_flights["arrival_time"]
    - silver_flights["departure_time"]
).dt.total_seconds() / 60

silver_flights["duration_minutes"] = (
    silver_flights["duration_minutes"].round(2)
)


# ------------------------------------------------------------
# 6. Identify overnight flights
# ------------------------------------------------------------

silver_flights["is_overnight"] = (
    silver_flights["arrival_time"].dt.date
    > silver_flights["departure_time"].dt.date
)


# ------------------------------------------------------------
# 7. Identify temporal anomalies
# ------------------------------------------------------------

silver_flights["is_anomaly"] = (
    silver_flights["arrival_time"]
    < silver_flights["departure_time"]
)


# ------------------------------------------------------------
# 8. Remove critical temporal anomalies
# ------------------------------------------------------------

anomaly_count = silver_flights["is_anomaly"].sum()

silver_flights = silver_flights[
    ~silver_flights["is_anomaly"]
].copy()


# ------------------------------------------------------------
# 9. Remove unnecessary source duration column
# ------------------------------------------------------------

silver_flights = silver_flights.drop(
    columns=["duration"],
    errors="ignore"
)


# ------------------------------------------------------------
# 10. Save Silver Flights
# ------------------------------------------------------------

silver_flights.to_csv(
    silver_path / "flights.csv",
    index=False
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("Duplicates removed:", removed_duplicates)
print("Temporal anomalies removed:", anomaly_count)
print("Overnight flights:",
      silver_flights["is_overnight"].sum())

print("Final Silver records:", len(silver_flights))

print("\nSilver columns:")
print(silver_flights.columns.tolist())

print("\nSaved to:")
print((silver_path / "flights.csv").resolve())

========== SILVER FLIGHTS ==========
Initial records: 1020
Duplicates removed: 15
Temporal anomalies removed: 1
Overnight flights: 122
Final Silver records: 1004

Silver columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration_minutes', 'is_overnight', 'is_anomaly']

Saved to:
C:\Users\User\OneDrive\Desktop\Airlines_DataEngineering\data\silver\flights.csv


In [50]:
# ============================================================
# STEP 6B — SILVER BOOKINGS
# ============================================================

silver_bookings = bookings.copy()

print("========== SILVER BOOKINGS ==========")
print("Initial records:", len(silver_bookings))


# ------------------------------------------------------------
# 1. Standardize column names
# ------------------------------------------------------------

silver_bookings.columns = (
    silver_bookings.columns
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 2. Standardize text fields
# ------------------------------------------------------------

for col in ["booking_id", "flight_id", "passenger_id", "status"]:
    if col in silver_bookings.columns:
        silver_bookings[col] = (
            silver_bookings[col]
            .astype("string")
            .str.strip()
            .str.upper()
        )


# ------------------------------------------------------------
# 3. Standardize booking status
# ------------------------------------------------------------

valid_statuses = {
    "CONFIRMED",
    "CANCELLED",
    "PENDING"
}

silver_bookings["status"] = (
    silver_bookings["status"]
    .where(
        silver_bookings["status"].isin(valid_statuses),
        "INVALID"
    )
)


# ------------------------------------------------------------
# 4. Remove duplicate booking IDs
# ------------------------------------------------------------

before = len(silver_bookings)

silver_bookings = silver_bookings.drop_duplicates(
    subset=["booking_id"],
    keep="first"
)

removed = before - len(silver_bookings)


# ------------------------------------------------------------
# 5. Add booking status flags
# ------------------------------------------------------------

silver_bookings["is_confirmed"] = (
    silver_bookings["status"] == "CONFIRMED"
)

silver_bookings["is_cancelled"] = (
    silver_bookings["status"] == "CANCELLED"
)

silver_bookings["is_pending"] = (
    silver_bookings["status"] == "PENDING"
)


# ------------------------------------------------------------
# 6. Save Silver Bookings
# ------------------------------------------------------------

silver_bookings.to_csv(
    silver_path / "bookings.csv",
    index=False
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("Duplicate booking IDs removed:", removed)

print(
    "Invalid/missing statuses:",
    (~silver_bookings["status"].isin(valid_statuses)).sum()
)

print("Final Silver records:", len(silver_bookings))

print("\nStatus distribution:")
print(silver_bookings["status"].value_counts())

print("\nSaved to:")
print((silver_path / "bookings.csv").resolve())

========== SILVER BOOKINGS ==========
Initial records: 1000
Duplicate booking IDs removed: 0
Invalid/missing statuses: 75
Final Silver records: 1000

Status distribution:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
INVALID       75
Name: count, dtype: Int64

Saved to:
C:\Users\User\OneDrive\Desktop\Airlines_DataEngineering\data\silver\bookings.csv


In [52]:
# ============================================================
# STEP 6C — SILVER PASSENGERS
# ============================================================

import hashlib

silver_passengers = passengers.copy()

print("========== SILVER PASSENGERS ==========")
print("Initial records:", len(silver_passengers))


# ------------------------------------------------------------
# 1. Standardize column names
# ------------------------------------------------------------

silver_passengers.columns = (
    silver_passengers.columns
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 2. Standardize names
# ------------------------------------------------------------

for col in ["first_name", "last_name"]:
    silver_passengers[col] = (
        silver_passengers[col]
        .astype("string")
        .str.strip()
        .str.title()
    )


# ------------------------------------------------------------
# 3. Standardize gender
# ------------------------------------------------------------

silver_passengers["gender"] = (
    silver_passengers["gender"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# 4. Handle missing last name
# ------------------------------------------------------------

silver_passengers["last_name"] = (
    silver_passengers["last_name"]
    .fillna("UNKNOWN")
)


# ------------------------------------------------------------
# 5. Validate Aadhaar
# ------------------------------------------------------------

aadhaar_clean = (
    silver_passengers["aadhaar_id"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
)

silver_passengers["aadhaar_valid"] = (
    aadhaar_clean.str.len() == 12
)


# ------------------------------------------------------------
# 6. Create hashed Aadhaar
# ------------------------------------------------------------

def hash_value(value):

    if pd.isna(value):
        return None

    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()


silver_passengers["aadhaar_hash"] = (
    silver_passengers["aadhaar_id"]
    .apply(hash_value)
)


# ------------------------------------------------------------
# 7. Create masked phone
# ------------------------------------------------------------

def mask_phone(value):

    if pd.isna(value):
        return None

    value = str(value)

    if len(value) >= 4:
        return "*" * (len(value) - 4) + value[-4:]

    return "****"


silver_passengers["phone_masked"] = (
    silver_passengers["phone"]
    .apply(mask_phone)
)


# ------------------------------------------------------------
# 8. Create hashed email
# ------------------------------------------------------------

silver_passengers["email_hash"] = (
    silver_passengers["email"]
    .apply(hash_value)
)


# ------------------------------------------------------------
# 9. Create age group
# ------------------------------------------------------------

silver_passengers["age_group"] = pd.cut(
    silver_passengers["age"],
    bins=[0, 18, 30, 45, 60, 100],
    labels=[
        "0-18",
        "19-30",
        "31-45",
        "46-60",
        "60+"
    ]
)


# ------------------------------------------------------------
# 10. Remove raw PII from analytical Silver
# ------------------------------------------------------------

silver_passengers = silver_passengers.drop(
    columns=[
        "aadhaar_id",
        "phone",
        "email"
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# 11. Remove duplicate passenger records
# ------------------------------------------------------------

# We do NOT blindly remove all duplicate passenger IDs,
# because some duplicates may contain corrected information.

# Exact duplicates are safe to remove.
silver_passengers = silver_passengers.drop_duplicates()


# ------------------------------------------------------------
# 12. Save Silver Passengers
# ------------------------------------------------------------

silver_passengers.to_csv(
    silver_path / "passengers.csv",
    index=False
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print(
    "Invalid Aadhaar records:",
    (~silver_passengers["aadhaar_valid"]).sum()
)

print(
    "Final Silver records:",
    len(silver_passengers)
)

print("\nPII-protected columns:")
print([
    "aadhaar_hash",
    "phone_masked",
    "email_hash"
])

print("\nSaved to:")
print((silver_path / "passengers.csv").resolve())

========== SILVER PASSENGERS ==========
Initial records: 1039
Invalid Aadhaar records: 114
Final Silver records: 1039

PII-protected columns:
['aadhaar_hash', 'phone_masked', 'email_hash']

Saved to:
C:\Users\User\OneDrive\Desktop\Airlines_DataEngineering\data\silver\passengers.csv


In [51]:
# ============================================================
# STEP 6D — SILVER PAYMENTS
# ============================================================

silver_payments = payments.copy()

print("========== SILVER PAYMENTS ==========")
print("Initial records:", len(silver_payments))


# ------------------------------------------------------------
# 1. Standardize column names
# ------------------------------------------------------------

silver_payments.columns = (
    silver_payments.columns
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 2. Standardize text fields
# ------------------------------------------------------------

for col in ["payment_id", "booking_id", "payment_method"]:

    silver_payments[col] = (
        silver_payments[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )


# ------------------------------------------------------------
# 3. Convert amount to numeric
# ------------------------------------------------------------

silver_payments["amount"] = pd.to_numeric(
    silver_payments["amount"],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. Create payment validity flag
# ------------------------------------------------------------

silver_payments["amount_valid"] = (
    silver_payments["amount"].notna()
)


# ------------------------------------------------------------
# 5. Standardize payment method
# ------------------------------------------------------------

valid_methods = {
    "UPI",
    "CARD",
    "NETBANKING"
}

silver_payments["payment_method"] = (
    silver_payments["payment_method"]
    .where(
        silver_payments["payment_method"]
        .isin(valid_methods),
        "UNKNOWN"
    )
)


# ------------------------------------------------------------
# 6. Remove exact duplicate records
# ------------------------------------------------------------

silver_payments = (
    silver_payments.drop_duplicates()
)


# ------------------------------------------------------------
# 7. Create payment status
# ------------------------------------------------------------

silver_payments["payment_status"] = (
    silver_payments["amount"]
    .notna()
    .map({
        True: "VALID",
        False: "INVALID"
    })
)


# ------------------------------------------------------------
# 8. Save Silver Payments
# ------------------------------------------------------------

silver_payments.to_csv(
    silver_path / "payments.csv",
    index=False
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print(
    "Invalid/missing payment amounts:",
    (~silver_payments["amount_valid"]).sum()
)

print(
    "Valid payment amounts:",
    silver_payments["amount_valid"].sum()
)

print(
    "Final Silver records:",
    len(silver_payments)
)

print("\nPayment status:")
print(
    silver_payments["payment_status"].value_counts()
)

print("\nSaved to:")
print((silver_path / "payments.csv").resolve())

========== SILVER PAYMENTS ==========
Initial records: 1000
Invalid/missing payment amounts: 78
Valid payment amounts: 922
Final Silver records: 1000

Payment status:
payment_status
VALID      922
INVALID     78
Name: count, dtype: int64

Saved to:
C:\Users\User\OneDrive\Desktop\Airlines_DataEngineering\data\silver\payments.csv


In [54]:
print("========== GOLD INPUT SCHEMAS ==========\n")

print("FLIGHTS COLUMNS:")
print(silver_flights.columns.tolist())

print("\nBOOKINGS COLUMNS:")
print(silver_bookings.columns.tolist())

print("\nPASSENGERS COLUMNS:")
print(silver_passengers.columns.tolist())

print("\nPAYMENTS COLUMNS:")
print(silver_payments.columns.tolist())

========== GOLD INPUT SCHEMAS ==========

FLIGHTS COLUMNS:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration_minutes', 'is_overnight', 'is_anomaly']

BOOKINGS COLUMNS:
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone', 'is_confirmed', 'is_cancelled', 'is_pending']

PASSENGERS COLUMNS:
['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'date_of_birth', 'aadhaar_valid', 'aadhaar_hash', 'phone_masked', 'email_hash', 'age_group']

PAYMENTS COLUMNS:
['payment_id', 'booking_id', 'amount', 'payment_method', 'amount_valid', 'payment_status']


In [55]:
# ============================================
# GOLD - DIM AIRLINE
# ============================================

dim_airline = (
    silver_flights[["airline"]]
    .drop_duplicates()
    .sort_values("airline")
    .reset_index(drop=True)
)

# Create surrogate key
dim_airline.insert(0, "airline_key", range(1, len(dim_airline) + 1))

# Save
dim_airline.to_csv(
    gold_path / "dim_airline.csv",
    index=False
)

print("========== DIM AIRLINE ==========")
print(dim_airline)
print("\nNumber of airlines:", len(dim_airline))
print("\nSaved to:", gold_path / "dim_airline.csv")

========== DIM AIRLINE ==========
   airline_key    airline
0            1  AIR INDIA
1            2     INDIGO
2            3   SPICEJET
3            4    UNKNOWN
4            5    VISTARA

Number of airlines: 5

Saved to: ..\data\gold\dim_airline.csv


In [56]:
# ============================================
# GOLD - DIM ROUTE
# ============================================

dim_route = (
    silver_flights[["source", "destination"]]
    .drop_duplicates()
    .sort_values(["source", "destination"])
    .reset_index(drop=True)
)

# Create a readable route name
dim_route["route"] = (
    dim_route["source"] + " → " + dim_route["destination"]
)

# Create surrogate key
dim_route.insert(0, "route_key", range(1, len(dim_route) + 1))

# Reorder columns
dim_route = dim_route[
    ["route_key", "route", "source", "destination"]
]

# Save
dim_route.to_csv(
    gold_path / "dim_route.csv",
    index=False
)

print("========== DIM ROUTE ==========")
print(dim_route.head(20))

print("\nNumber of unique routes:", len(dim_route))

print("\nSaved to:", gold_path / "dim_route.csv")

========== DIM ROUTE ==========
    route_key      route source destination
0           1  BLR → BOM    BLR         BOM
1           2  BLR → CCU    BLR         CCU
2           3  BLR → DEL    BLR         DEL
3           4  BLR → HYD    BLR         HYD
4           5  BLR → MAA    BLR         MAA
5           6  BOM → BLR    BOM         BLR
6           7  BOM → CCU    BOM         CCU
7           8  BOM → DEL    BOM         DEL
8           9  BOM → HYD    BOM         HYD
9          10  BOM → MAA    BOM         MAA
10         11  CCU → BLR    CCU         BLR
11         12  CCU → BOM    CCU         BOM
12         13  CCU → DEL    CCU         DEL
13         14  CCU → HYD    CCU         HYD
14         15  CCU → MAA    CCU         MAA
15         16  DEL → BLR    DEL         BLR
16         17  DEL → BOM    DEL         BOM
17         18  DEL → CCU    DEL         CCU
18         19  DEL → HYD    DEL         HYD
19         20  DEL → MAA    DEL         MAA

Number of unique routes: 30

Saved to: ..\d

In [57]:
# ============================================
# GOLD - DIM PASSENGER
# ============================================

passenger_gold = silver_passengers.copy()

# Rank records so we choose the best available
# record when a passenger_id appears more than once.
passenger_gold["_last_name_known"] = (
    passenger_gold["last_name"].notna() &
    (passenger_gold["last_name"].astype(str).str.upper() != "UNKNOWN")
)

passenger_gold["_quality_rank"] = (
    passenger_gold["aadhaar_valid"].astype(int) * 2
    + passenger_gold["_last_name_known"].astype(int)
)

# Keep the highest-quality record for each passenger_id
passenger_gold = (
    passenger_gold
    .sort_values(
        ["passenger_id", "_quality_rank"],
        ascending=[True, False]
    )
    .drop_duplicates("passenger_id", keep="first")
    .reset_index(drop=True)
)

# Select only analytical / protected fields
dim_passenger = passenger_gold[
    [
        "passenger_id",
        "first_name",
        "last_name",
        "age",
        "gender",
        "date_of_birth",
        "age_group",
        "aadhaar_valid",
        "aadhaar_hash",
        "phone_masked",
        "email_hash"
    ]
].copy()

# Create surrogate key
dim_passenger.insert(
    0,
    "passenger_key",
    range(1, len(dim_passenger) + 1)
)

# Save
dim_passenger.to_csv(
    gold_path / "dim_passenger.csv",
    index=False
)

print("========== DIM PASSENGER ==========")
print(dim_passenger.head(10))

print("\nUnique passengers:", len(dim_passenger))
print(
    "Duplicate passenger IDs remaining:",
    dim_passenger["passenger_id"].duplicated().sum()
)

print(
    "\nPII columns removed:",
    ["raw email", "raw phone", "raw Aadhaar"]
)

print("\nSaved to:", gold_path / "dim_passenger.csv")

========== DIM PASSENGER ==========
   passenger_key passenger_id first_name   last_name  age gender  \
0              1        P1000     Vivaan  Chatterjee   52      F   
1              2        P1001    Krishna       Reddy   15      M   
2              3        P1002       Myra       Naidu   72      M   
3              4        P1003       Myra      Mishra   61      F   
4              5        P1004     Saanvi    Banerjee   21      M   
5              6        P1005    Reyansh       Joshi   83      M   
6              7        P1006      Kavya        Nair   87      M   
7              8        P1007       Sara       Joshi   75      M   
8              9        P1008      Kavya      Pillai   75      M   
9             10        P1009        Sai    Kulkarni   88      M   

  date_of_birth age_group  aadhaar_valid  \
0    1974-04-08     46-60           True   
1    2011-03-07      0-18           True   
2    1954-09-10       60+           True   
3    1965-03-12       60+           Tru

In [58]:
# ============================================
# GOLD - DIM PAYMENT METHOD
# ============================================

dim_payment_method = (
    silver_payments[["payment_method"]]
    .drop_duplicates()
    .sort_values("payment_method")
    .reset_index(drop=True)
)

# Create surrogate key
dim_payment_method.insert(
    0,
    "payment_method_key",
    range(1, len(dim_payment_method) + 1)
)

# Save
dim_payment_method.to_csv(
    gold_path / "dim_payment_method.csv",
    index=False
)

print("========== DIM PAYMENT METHOD ==========")
print(dim_payment_method)

print("\nNumber of payment methods:", len(dim_payment_method))

print(
    "\nSaved to:",
    gold_path / "dim_payment_method.csv"
)

========== DIM PAYMENT METHOD ==========
   payment_method_key payment_method
0                   1           CARD
1                   2     NETBANKING
2                   3            UPI

Number of payment methods: 3

Saved to: ..\data\gold\dim_payment_method.csv


In [59]:
# ============================================
# GOLD - DIM DATE
# ============================================

# Get the minimum and maximum dates from flight data
min_date = silver_flights["departure_time"].min().date()
max_date = silver_flights["arrival_time"].max().date()

# Create continuous date range
date_range = pd.date_range(
    start=min_date,
    end=max_date,
    freq="D"
)

dim_date = pd.DataFrame({
    "date": date_range
})

# Date attributes
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.day_name()
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["year"] = dim_date["date"].dt.year
dim_date["week_of_year"] = dim_date["date"].dt.isocalendar().week.astype(int)

# Reorder columns
dim_date = dim_date[
    [
        "date_key",
        "date",
        "day",
        "day_name",
        "month",
        "month_name",
        "quarter",
        "year",
        "week_of_year"
    ]
]

# Save
dim_date.to_csv(
    gold_path / "dim_date.csv",
    index=False
)

print("========== DIM DATE ==========")
print(dim_date.head())
print("\n...")
print(dim_date.tail())

print("\nNumber of dates:", len(dim_date))
print("Date range:", min_date, "to", max_date)

print("\nSaved to:", gold_path / "dim_date.csv")

========== DIM DATE ==========
   date_key       date  day  day_name  month month_name quarter  year  \
0  20260417 2026-04-17   17    Friday      4      April      Q2  2026   
1  20260418 2026-04-18   18  Saturday      4      April      Q2  2026   
2  20260419 2026-04-19   19    Sunday      4      April      Q2  2026   
3  20260420 2026-04-20   20    Monday      4      April      Q2  2026   
4  20260421 2026-04-21   21   Tuesday      4      April      Q2  2026   

   week_of_year  
0            16  
1            16  
2            16  
3            17  
4            17  

...
   date_key       date  day  day_name  month month_name quarter  year  \
0  20260417 2026-04-17   17    Friday      4      April      Q2  2026   
1  20260418 2026-04-18   18  Saturday      4      April      Q2  2026   
2  20260419 2026-04-19   19    Sunday      4      April      Q2  2026   
3  20260420 2026-04-20   20    Monday      4      April      Q2  2026   
4  20260421 2026-04-21   21   Tuesday      4      Ap

In [60]:
# ============================================
# GOLD - FACT TABLES
# ============================================

print("========== CREATING FACT TABLES ==========\n")

# ------------------------------------------------
# 1. FACT FLIGHT
# ------------------------------------------------

fact_flight = silver_flights.copy()

# Join airline dimension
fact_flight = fact_flight.merge(
    dim_airline,
    on="airline",
    how="left"
)

# Join route dimension
fact_flight = fact_flight.merge(
    dim_route[["route_key", "source", "destination"]],
    on=["source", "destination"],
    how="left"
)

# Create surrogate flight key
fact_flight = fact_flight.reset_index(drop=True)
fact_flight.insert(
    0,
    "flight_key",
    range(1, len(fact_flight) + 1)
)

# Create date keys
fact_flight["departure_date_key"] = (
    fact_flight["departure_time"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

fact_flight["arrival_date_key"] = (
    fact_flight["arrival_time"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# Select final columns
fact_flight = fact_flight[
    [
        "flight_key",
        "flight_id",
        "airline_key",
        "route_key",
        "departure_date_key",
        "arrival_date_key",
        "departure_time",
        "arrival_time",
        "duration_minutes",
        "is_overnight",
        "is_anomaly"
    ]
]

# Save
fact_flight.to_csv(
    gold_path / "fact_flight.csv",
    index=False
)


# ------------------------------------------------
# 2. FACT BOOKING
# ------------------------------------------------

fact_booking = silver_bookings.copy()

# Join passenger dimension
fact_booking = fact_booking.merge(
    dim_passenger[["passenger_key", "passenger_id"]],
    on="passenger_id",
    how="left"
)

# Create booking date key
fact_booking["booking_date"] = pd.to_datetime(
    fact_booking["booking_date"],
    errors="coerce"
)

fact_booking["booking_date_key"] = (
    fact_booking["booking_date"]
    .dt.strftime("%Y%m%d")
)

# Convert valid dates to integer, keep missing as NaN
fact_booking["booking_date_key"] = pd.to_numeric(
    fact_booking["booking_date_key"],
    errors="coerce"
)

# Create surrogate booking key
fact_booking = fact_booking.reset_index(drop=True)

fact_booking.insert(
    0,
    "booking_key",
    range(1, len(fact_booking) + 1)
)

# Protect PII:
# passport number and emergency contact information
# are NOT included in the Gold analytical fact table.

fact_booking = fact_booking[
    [
        "booking_key",
        "booking_id",
        "flight_id",
        "passenger_key",
        "booking_date_key",
        "status",
        "seat_number",
        "is_confirmed",
        "is_cancelled",
        "is_pending"
    ]
]

# Save
fact_booking.to_csv(
    gold_path / "fact_booking.csv",
    index=False
)


# ------------------------------------------------
# 3. FACT PAYMENT
# ------------------------------------------------

fact_payment = silver_payments.copy()

# Join payment method dimension
fact_payment = fact_payment.merge(
    dim_payment_method,
    on="payment_method",
    how="left"
)

# Create booking relationship
fact_payment = fact_payment.merge(
    fact_booking[["booking_key", "booking_id"]],
    on="booking_id",
    how="left"
)

# Create surrogate payment key
fact_payment = fact_payment.reset_index(drop=True)

fact_payment.insert(
    0,
    "payment_key",
    range(1, len(fact_payment) + 1)
)

# Select final columns
fact_payment = fact_payment[
    [
        "payment_key",
        "payment_id",
        "booking_key",
        "payment_method_key",
        "amount",
        "amount_valid",
        "payment_status"
    ]
]

# Save
fact_payment.to_csv(
    gold_path / "fact_payment.csv",
    index=False
)


# ============================================
# VERIFICATION
# ============================================

print("FACT FLIGHT")
print("Rows:", len(fact_flight))
print("Columns:", fact_flight.columns.tolist())

print("\nFACT BOOKING")
print("Rows:", len(fact_booking))
print("Columns:", fact_booking.columns.tolist())

print("\nFACT PAYMENT")
print("Rows:", len(fact_payment))
print("Columns:", fact_payment.columns.tolist())

print("\n========== GOLD FACT TABLES SAVED ==========")
print("fact_flight.csv")
print("fact_booking.csv")
print("fact_payment.csv")

========== CREATING FACT TABLES ==========

FACT FLIGHT
Rows: 1004
Columns: ['flight_key', 'flight_id', 'airline_key', 'route_key', 'departure_date_key', 'arrival_date_key', 'departure_time', 'arrival_time', 'duration_minutes', 'is_overnight', 'is_anomaly']

FACT BOOKING
Rows: 1000
Columns: ['booking_key', 'booking_id', 'flight_id', 'passenger_key', 'booking_date_key', 'status', 'seat_number', 'is_confirmed', 'is_cancelled', 'is_pending']

FACT PAYMENT
Rows: 1000
Columns: ['payment_key', 'payment_id', 'booking_key', 'payment_method_key', 'amount', 'amount_valid', 'payment_status']

========== GOLD FACT TABLES SAVED ==========
fact_flight.csv
fact_booking.csv
fact_payment.csv


In [62]:
# ============================================
# ASG AIRLINES - PIPELINE ORCHESTRATION
# ============================================

from datetime import datetime
import time
import pandas as pd

pipeline_log = []

pipeline_start = datetime.now()

print("=" * 60)
print("        ASG AIRLINES DATA PIPELINE")
print("=" * 60)
print("Pipeline started:", pipeline_start.strftime("%Y-%m-%d %H:%M:%S"))
print()


def log_stage(stage, status, message, rows=None, start_time=None):
    """Record pipeline stage execution details."""

    end_time = time.time()

    duration = None
    if start_time is not None:
        duration = round(end_time - start_time, 3)

    pipeline_log.append({
        "pipeline_run_time": pipeline_start,
        "stage": stage,
        "status": status,
        "rows": rows,
        "duration_seconds": duration,
        "message": message
    })

    symbol = " " if status == "SUCCESS" else "✗"

    print(
        f"[{symbol}] {stage:<25} "
        f"{status:<10} "
        f"Rows: {rows if rows is not None else 'N/A'}"
    )

    if message:
        print("    ", message)


# ============================================
# STAGE 1 - BRONZE
# ============================================

stage_start = time.time()

try:

    bronze_files = list(bronze_path.glob("*.csv"))

    if len(bronze_files) == 0:
        raise FileNotFoundError(
            "No files found in Bronze layer."
        )

    log_stage(
        "Bronze Ingestion",
        "SUCCESS",
        f"{len(bronze_files)} source files available.",
        len(bronze_files),
        stage_start
    )

except Exception as e:

    log_stage(
        "Bronze Ingestion",
        "FAILED",
        str(e),
        None,
        stage_start
    )


# ============================================
# STAGE 2 - DATA QUALITY
# ============================================

stage_start = time.time()

try:

    dq_file = project_root / "output" / "dq_report.csv"

    if not dq_file.exists():
        raise FileNotFoundError(
            "DQ report not found."
        )

    dq_report = pd.read_csv(dq_file)

    log_stage(
        "Data Quality",
        "SUCCESS",
        "DQ report successfully generated.",
        len(dq_report),
        stage_start
    )

except Exception as e:

    log_stage(
        "Data Quality",
        "FAILED",
        str(e),
        None,
        stage_start
    )


# ============================================
# STAGE 3 - QUARANTINE
# ============================================

stage_start = time.time()

try:

    quarantine_files = list(
        quarantine_path.glob("*.csv")
    )

    quarantine_rows = 0

    for file in quarantine_files:

        df = pd.read_csv(file)
        quarantine_rows += len(df)

    log_stage(
        "Quarantine",
        "SUCCESS",
        f"{len(quarantine_files)} quarantine files processed.",
        quarantine_rows,
        stage_start
    )

except Exception as e:

    log_stage(
        "Quarantine",
        "FAILED",
        str(e),
        None,
        stage_start
    )


# ============================================
# STAGE 4 - SILVER
# ============================================

stage_start = time.time()

try:

    silver_files = list(
        silver_path.glob("*.csv")
    )

    if len(silver_files) == 0:
        raise FileNotFoundError(
            "No Silver datasets found."
        )

    silver_rows = sum(
        len(pd.read_csv(file))
        for file in silver_files
    )

    log_stage(
        "Silver Transformation",
        "SUCCESS",
        f"{len(silver_files)} cleaned datasets available.",
        silver_rows,
        stage_start
    )

except Exception as e:

    log_stage(
        "Silver Transformation",
        "FAILED",
        str(e),
        None,
        stage_start
    )


# ============================================
# STAGE 5 - GOLD
# ============================================

stage_start = time.time()

try:

    gold_files = list(
        gold_path.glob("*.csv")
    )

    if len(gold_files) == 0:
        raise FileNotFoundError(
            "No Gold datasets found."
        )

    gold_rows = sum(
        len(pd.read_csv(file))
        for file in gold_files
    )

    log_stage(
        "Gold Data Modelling",
        "SUCCESS",
        f"{len(gold_files)} Gold datasets available.",
        gold_rows,
        stage_start
    )

except Exception as e:

    log_stage(
        "Gold Data Modelling",
        "FAILED",
        str(e),
        None,
        stage_start
    )


# ============================================
# PIPELINE SUMMARY
# ============================================

pipeline_end = datetime.now()

pipeline_log_df = pd.DataFrame(pipeline_log)

print()
print("=" * 60)
print("              PIPELINE SUMMARY")
print("=" * 60)

print("Started :", pipeline_start.strftime("%Y-%m-%d %H:%M:%S"))
print("Finished:", pipeline_end.strftime("%Y-%m-%d %H:%M:%S"))

print("\nStage Status:")
print(
    pipeline_log_df[
        ["stage", "status", "rows", "duration_seconds"]
    ].to_string(index=False)
)

# Save pipeline execution log
pipeline_log_path = (
    project_root / "output" / "pipeline_log.csv"
)

pipeline_log_df.to_csv(
    pipeline_log_path,
    index=False
)

print("\nPipeline log saved to:")
print(pipeline_log_path)

# Overall status
if (pipeline_log_df["status"] == "FAILED").any():

    print("\nPIPELINE COMPLETED WITH ERRORS")

else:

    print("\nPIPELINE COMPLETED SUCCESSFULLY")

        ASG AIRLINES DATA PIPELINE
Pipeline started: 2026-09-05 19:13:23

[ ] Bronze Ingestion          SUCCESS    Rows: 4
     4 source files available.
[ ] Data Quality              SUCCESS    Rows: 23
     DQ report successfully generated.
[ ] Quarantine                SUCCESS    Rows: 79
     2 quarantine files processed.
[ ] Silver Transformation     SUCCESS    Rows: 4043
     4 cleaned datasets available.
[ ] Gold Data Modelling       SUCCESS    Rows: 4047
     8 Gold datasets available.

              PIPELINE SUMMARY
Started : 2026-09-05 19:13:23
Finished: 2026-09-05 19:13:23

Stage Status:
                stage  status  rows  duration_seconds
     Bronze Ingestion SUCCESS     4             0.001
         Data Quality SUCCESS    23             0.004
           Quarantine SUCCESS    79             0.006
Silver Transformation SUCCESS  4043             0.028
  Gold Data Modelling SUCCESS  4047             0.020

Pipeline log saved to:
..\output\pipeline_log.csv

PIPELINE COMPLETED

In [63]:
# ============================================
# SCHEMA VALIDATION GATE
# ============================================

EXPECTED_SCHEMAS = {

    "flights": [
        "flight_id",
        "airline",
        "source",
        "destination",
        "departure_time",
        "arrival_time",
        "duration"
    ],

    "bookings": [
        "booking_id",
        "passenger_id",
        "flight_id",
        "booking_date",
        "status",
        "passport_number",
        "seat_number",
        "emergency_contact_name",
        "emergency_contact_phone"
    ],

    "passengers": [
        "passenger_id",
        "first_name",
        "last_name",
        "age",
        "gender",
        "email",
        "phone",
        "aadhaar_id",
        "date_of_birth"
    ],

    "payments": [
        "payment_id",
        "booking_id",
        "amount",
        "payment_method"
    ]
}


def validate_schema(df, dataset_name):

    expected = EXPECTED_SCHEMAS[dataset_name]

    actual = list(df.columns)

    missing_columns = [
        col for col in expected
        if col not in actual
    ]

    unexpected_columns = [
        col for col in actual
        if col not in expected
    ]

    if missing_columns or unexpected_columns:

        return {
            "dataset": dataset_name,
            "status": "FAILED",
            "missing_columns": missing_columns,
            "unexpected_columns": unexpected_columns
        }

    return {
        "dataset": dataset_name,
        "status": "SUCCESS",
        "missing_columns": [],
        "unexpected_columns": []
    }

In [64]:
# ============================================
# RUN SCHEMA VALIDATION
# ============================================

schema_results = []

for dataset_name in EXPECTED_SCHEMAS:

    file_path = bronze_path / f"{dataset_name}.csv"

    df = pd.read_csv(file_path)

    result = validate_schema(
        df,
        dataset_name
    )

    schema_results.append(result)

    print(
        f"{dataset_name:<12} "
        f"{result['status']}"
    )

    if result["missing_columns"]:
        print(
            "  Missing:",
            result["missing_columns"]
        )

    if result["unexpected_columns"]:
        print(
            "  Unexpected:",
            result["unexpected_columns"]
        )


schema_results_df = pd.DataFrame(schema_results)

schema_results_df

flights      SUCCESS
bookings     SUCCESS
passengers   SUCCESS
payments     SUCCESS


,dataset,status,missing_columns,unexpected_columns
0,flights,SUCCESS,[],[]
1,bookings,SUCCESS,[],[]
2,passengers,SUCCESS,[],[]
3,payments,SUCCESS,[],[]


In [67]:
# ============================================
# REFERENTIAL INTEGRITY VALIDATION
# ============================================

flights_df = pd.read_csv(
    silver_path / "flights.csv"
)

bookings_df = pd.read_csv(
    silver_path / "bookings.csv"
)

passengers_df = pd.read_csv(
    silver_path / "passengers.csv"
)

payments_df = pd.read_csv(
    silver_path / "payments.csv"
)


# Bookings → Flights
missing_flights = bookings_df[
    ~bookings_df["flight_id"].isin(
        flights_df["flight_id"]
    )
]


# Bookings → Passengers
missing_passengers = bookings_df[
    ~bookings_df["passenger_id"].isin(
        passengers_df["passenger_id"]
    )
]


# Payments → Bookings
missing_bookings = payments_df[
    ~payments_df["booking_id"].isin(
        bookings_df["booking_id"]
    )
]


print("Referential Integrity Checks")
print("=" * 45)

print(
    "Bookings → Flights    :",
    len(missing_flights),
    "invalid references"
)

print(
    "Bookings → Passengers :",
    len(missing_passengers),
    "invalid references"
)

print(
    "Payments → Bookings   :",
    len(missing_bookings),
    "invalid references"
)

Referential Integrity Checks
Bookings → Flights    : 1 invalid references
Bookings → Passengers : 0 invalid references
Payments → Bookings   : 0 invalid references


In [68]:
# ============================================
# PIPELINE QUALITY GATE
# ============================================

schema_failed = (
    schema_results_df["status"] == "FAILED"
).any()

referential_integrity_failed = (
    len(missing_flights) > 0
    or len(missing_passengers) > 0
    or len(missing_bookings) > 0
)


if schema_failed:

    print("PIPELINE BLOCKED")
    print("Schema validation failed.")

elif referential_integrity_failed:

    print("PIPELINE BLOCKED")
    print("Referential integrity validation failed.")

else:

    print("PIPELINE QUALITY GATE PASSED")
    print("All validation checks passed.")

PIPELINE BLOCKED
Referential integrity validation failed.


In [70]:
# ============================================
# DEPENDENCY-AWARE REFERENTIAL INTEGRITY
# ============================================

quarantined_flights = set()

flight_quarantine_file = quarantine_path / "flights_quarantine.csv"

if flight_quarantine_file.exists():

    flight_q = pd.read_csv(
        flight_quarantine_file
    )

    if "flight_id" in flight_q.columns:
        quarantined_flights = set(
            flight_q["flight_id"]
            .dropna()
            .astype(str)
            .str.strip()
            .str.upper()
        )


# Normalize booking flight IDs
bookings_df["flight_id"] = (
    bookings_df["flight_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# Identify bookings referencing quarantined flights
bookings_with_quarantined_flights = bookings_df[
    bookings_df["flight_id"].isin(
        quarantined_flights
    )
].copy()


print("=" * 60)
print("DEPENDENCY-AWARE VALIDATION")
print("=" * 60)

print(
    "Quarantined flight IDs:",
    len(quarantined_flights)
)

print(
    "Bookings referencing quarantined flights:",
    len(bookings_with_quarantined_flights)
)

if len(bookings_with_quarantined_flights) > 0:

    print("\nAffected bookings:")

    print(
        bookings_with_quarantined_flights[
            ["booking_id", "flight_id"]
        ].to_string(index=False)
    )

DEPENDENCY-AWARE VALIDATION
Quarantined flight IDs: 0
Bookings referencing quarantined flights: 0


In [72]:
# ============================================
# READ QUARANTINED FLIGHTS
# ============================================

flight_quarantine_file = quarantine_path / "flights.csv"

print("Flight quarantine file:", flight_quarantine_file)
print("Exists:", flight_quarantine_file.exists())

flight_q = pd.read_csv(flight_quarantine_file)

print("\nColumns:")
print(flight_q.columns.tolist())

print("\nQuarantined flights:")
print(
    flight_q[["flight_id"]].to_string(index=False)
)

Flight quarantine file: ..\data\quarantine\flights.csv
Exists: True

Columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration', 'dq_rule', 'dq_severity', 'dq_reason']

Quarantined flights:
flight_id
    SJ192


In [74]:
# ============================================
# DEPENDENCY-AWARE VALIDATION
# ============================================

quarantined_flights = set(
    flight_q["flight_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

bookings_df["flight_id"] = (
    bookings_df["flight_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

bookings_with_quarantined_flights = bookings_df[
    bookings_df["flight_id"].isin(quarantined_flights)
].copy()

print("=" * 60)
print("DEPENDENCY-AWARE VALIDATION")
print("=" * 60)

print("Quarantined flight IDs:", len(quarantined_flights))
print(
    "Bookings referencing quarantined flights:",
    len(bookings_with_quarantined_flights)
)

if len(bookings_with_quarantined_flights) > 0:
    print("\nAffected bookings:")
    print(
        bookings_with_quarantined_flights[
            ["booking_id", "flight_id"]
        ].to_string(index=False)
    )

DEPENDENCY-AWARE VALIDATION
Quarantined flight IDs: 1
Bookings referencing quarantined flights: 1

Affected bookings:
booking_id flight_id
     B1636     SJ192


In [76]:
# ============================================
# ADD UPSTREAM QUARANTINE DEPENDENCY FLAG
# ============================================

bookings_df["flight_quarantined"] = (
    bookings_df["flight_id"].isin(quarantined_flights)
)

bookings_df["flight_reference_status"] = "VALID"

bookings_df.loc[
    bookings_df["flight_quarantined"],
    "flight_reference_status"
] = "QUARANTINED_PARENT"


print("=" * 60)
print("BOOKING → FLIGHT REFERENCE STATUS")
print("=" * 60)

print(
    bookings_df["flight_reference_status"].value_counts()
)

print("\nAffected booking:")

print(
    bookings_df[
        bookings_df["flight_quarantined"]
    ][
        [
            "booking_id",
            "flight_id",
            "flight_quarantined",
            "flight_reference_status"
        ]
    ].to_string(index=False)
)
bookings_df.to_csv(
    silver_path / "bookings.csv",
    index=False
)

print("\nSilver bookings updated successfully.")

BOOKING → FLIGHT REFERENCE STATUS
flight_reference_status
VALID                 999
QUARANTINED_PARENT      1
Name: count, dtype: int64

Affected booking:
booking_id flight_id  flight_quarantined flight_reference_status
     B1636     SJ192                True      QUARANTINED_PARENT

Silver bookings updated successfully.


In [78]:
# ============================================
# FINAL REFERENTIAL INTEGRITY QUALITY GATE
# ============================================

true_invalid_flight_refs = missing_flights[
    ~missing_flights["flight_id"].isin(
        quarantined_flights
    )
]

referential_integrity_failed = (
    len(true_invalid_flight_refs) > 0
    or
    len(missing_passengers) > 0
    or
    len(missing_bookings) > 0
)

dependency_warnings = (
    len(bookings_with_quarantined_flights)
)


print("=" * 60)
print("FINAL REFERENTIAL INTEGRITY QUALITY GATE")
print("=" * 60)


if referential_integrity_failed:

    print(" FAILED")
    print(
        "Unresolved referential integrity violations detected."
    )

elif dependency_warnings > 0:

    print("PASSED WITH WARNING")
    print(
        f"{dependency_warnings} booking(s) reference "
        "quarantined parent flight record(s)."
    )

else:

    print(" PASSED")
    print(
        "All referential integrity checks passed."
    )

FINAL REFERENTIAL INTEGRITY QUALITY GATE
PASSED WITH WARNING
1 booking(s) reference quarantined parent flight record(s).


In [79]:
# ============================================
# INSPECT DUPLICATE FLIGHT IDs
# ============================================

flights_df = pd.read_csv(
    silver_path / "flights.csv"
)

duplicate_flights = (
    flights_df[
        flights_df["flight_id"].duplicated(
            keep=False
        )
    ]
    .sort_values("flight_id")
)

print("=" * 60)
print("DUPLICATE FLIGHT ID ANALYSIS")
print("=" * 60)

print(
    "Total flight rows:",
    len(flights_df)
)

print(
    "Unique flight IDs:",
    flights_df["flight_id"].nunique()
)

print(
    "Rows belonging to duplicated IDs:",
    len(duplicate_flights)
)

print(
    "Number of duplicated flight IDs:",
    duplicate_flights["flight_id"].nunique()
)

print("\nDuplicate flight records:")

print(
    duplicate_flights.to_string(index=False)
)

DUPLICATE FLIGHT ID ANALYSIS
Total flight rows: 1004
Unique flight IDs: 1003
Rows belonging to duplicated IDs: 2
Number of duplicated flight IDs: 1

Duplicate flight records:
flight_id airline source destination          departure_time            arrival_time  duration_minutes  is_overnight  is_anomaly
    6F250 UNKNOWN    DEL         BLR 2026-04-20 03:26:41.701 2026-04-20 07:30:41.701             244.0         False       False
    6F250 UNKNOWN    CCU         BLR 2026-04-20 02:23:41.702 2026-04-20 02:56:41.702              33.0         False       False


In [80]:
# ============================================
# CHECK AMBIGUOUS FLIGHT REFERENCES
# ============================================

ambiguous_flight_id = "6F250"

affected_bookings = bookings_df[
    bookings_df["flight_id"] == ambiguous_flight_id
].copy()

print("=" * 60)
print("AMBIGUOUS FLIGHT REFERENCE CHECK")
print("=" * 60)

print(
    "Bookings referencing 6F250:",
    len(affected_bookings)
)

if len(affected_bookings) > 0:
    print("\nAffected bookings:")
    print(
        affected_bookings.to_string(index=False)
    )
else:
    print(
        "\nNo bookings reference the ambiguous flight ID."
    )

AMBIGUOUS FLIGHT REFERENCE CHECK
Bookings referencing 6F250: 2

Affected bookings:
booking_id passenger_id flight_id            booking_date    status passport_number seat_number emergency_contact_name emergency_contact_phone  is_confirmed  is_cancelled  is_pending  flight_quarantined flight_reference_status
     B1235        P1979     6F250 2025-12-23 11:37:36.951   PENDING        Y7698731         13D          Vedhika Khare          +91-9835680345         False         False        True               False                   VALID
     B1294        P1898     6F250 2025-07-13 11:37:36.951 CANCELLED        J2383972         34B            Hitesh Sant          +91-7709899502         False          True       False               False                   VALID


In [84]:
# ============================================================
# STEP 8 — BUSINESS KPI / AGGREGATION LAYER
# ============================================================

import pandas as pd
from pathlib import Path

print("ASG AIRLINES — KPI & AGGREGATION LAYER")

# ------------------------------------------------------------
# 1. CREATE AGGREGATION DIRECTORY
# ------------------------------------------------------------

aggregation_path = gold_path / "aggregations"
aggregation_path.mkdir(parents=True, exist_ok=True)

print("\nAggregation directory:", aggregation_path)


# ------------------------------------------------------------
# 2. LOAD SILVER DATA
# ------------------------------------------------------------

flights = pd.read_csv(silver_path / "flights.csv")
bookings = pd.read_csv(silver_path / "bookings.csv")
passengers = pd.read_csv(silver_path / "passengers.csv")
payments = pd.read_csv(silver_path / "payments.csv")


# Convert numeric columns
flights["duration_minutes"] = pd.to_numeric(
    flights["duration_minutes"],
    errors="coerce"
)

payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)


# ============================================================
# A. FLIGHT KPI
# ============================================================

total_flights = len(flights)

flight_kpi = pd.DataFrame({
    "metric": [
        "Total Flights",
        "Average Flight Duration (Minutes)",
        "Minimum Flight Duration (Minutes)",
        "Maximum Flight Duration (Minutes)",
        "Overnight Flights",
        "Anomalous Flights"
    ],
    "value": [
        total_flights,
        round(flights["duration_minutes"].mean(), 2),
        round(flights["duration_minutes"].min(), 2),
        round(flights["duration_minutes"].max(), 2),
        int(flights["is_overnight"].sum()),
        int(flights["is_anomaly"].sum())
    ]
})

flight_kpi.to_csv(
    aggregation_path / "flight_kpis.csv",
    index=False
)


# ============================================================
# B. ROUTE KPI
# ============================================================

route_kpi = (
    flights
    .groupby(
        ["source", "destination"],
        dropna=False
    )
    .agg(
        total_flights=("flight_id", "count"),
        average_duration_minutes=(
            "duration_minutes",
            "mean"
        ),
        overnight_flights=(
            "is_overnight",
            "sum"
        ),
        anomalous_flights=(
            "is_anomaly",
            "sum"
        )
    )
    .reset_index()
)

route_kpi["average_duration_minutes"] = (
    route_kpi["average_duration_minutes"]
    .round(2)
)

route_kpi["route"] = (
    route_kpi["source"]
    + " - "
    + route_kpi["destination"]
)

route_kpi = route_kpi[
    [
        "route",
        "source",
        "destination",
        "total_flights",
        "average_duration_minutes",
        "overnight_flights",
        "anomalous_flights"
    ]
].sort_values(
    "total_flights",
    ascending=False
)

route_kpi.to_csv(
    aggregation_path / "route_kpis.csv",
    index=False
)


# ============================================================
# C. AIRLINE KPI
# ============================================================

airline_kpi = (
    flights
    .groupby("airline", dropna=False)
    .agg(
        total_flights=("flight_id", "count"),
        average_duration_minutes=(
            "duration_minutes",
            "mean"
        ),
        overnight_flights=(
            "is_overnight",
            "sum"
        ),
        anomalous_flights=(
            "is_anomaly",
            "sum"
        )
    )
    .reset_index()
)

airline_kpi["average_duration_minutes"] = (
    airline_kpi["average_duration_minutes"]
    .round(2)
)

airline_kpi["flight_share_percent"] = (
    airline_kpi["total_flights"]
    / total_flights
    * 100
).round(2)

airline_kpi["overnight_percent"] = (
    airline_kpi["overnight_flights"]
    / airline_kpi["total_flights"]
    * 100
).round(2)

airline_kpi = airline_kpi.sort_values(
    "total_flights",
    ascending=False
)

airline_kpi.to_csv(
    aggregation_path / "airline_kpis.csv",
    index=False
)


# ============================================================
# D. BOOKING KPI
# ============================================================

total_bookings = len(bookings)

confirmed = (
    bookings["is_confirmed"]
    .fillna(False)
    .sum()
)

cancelled = (
    bookings["is_cancelled"]
    .fillna(False)
    .sum()
)

pending = (
    bookings["is_pending"]
    .fillna(False)
    .sum()
)

booking_kpi = pd.DataFrame({
    "metric": [
        "Total Bookings",
        "Confirmed Bookings",
        "Cancelled Bookings",
        "Pending Bookings",
        "Confirmation Rate (%)",
        "Cancellation Rate (%)",
        "Pending Rate (%)",
        "Bookings With Quarantined Flight"
    ],
    "value": [
        total_bookings,
        int(confirmed),
        int(cancelled),
        int(pending),
        round(confirmed / total_bookings * 100, 2),
        round(cancelled / total_bookings * 100, 2),
        round(pending / total_bookings * 100, 2),
        int(
            bookings["flight_quarantined"]
            .fillna(False)
            .sum()
        )
        if "flight_quarantined" in bookings.columns
        else 0
    ]
})

booking_kpi.to_csv(
    aggregation_path / "booking_kpis.csv",
    index=False
)


# ============================================================
# E. PAYMENT KPI
# ============================================================

total_payments = len(payments)

valid_payments = (
    payments["amount_valid"]
    .fillna(False)
    .sum()
)

invalid_payments = total_payments - valid_payments

total_revenue = payments.loc[
    payments["amount_valid"].fillna(False),
    "amount"
].sum()

average_payment = payments.loc[
    payments["amount_valid"].fillna(False),
    "amount"
].mean()

payment_kpi = pd.DataFrame({
    "metric": [
        "Total Payment Transactions",
        "Valid Payment Transactions",
        "Invalid Payment Transactions",
        "Total Revenue",
        "Average Valid Transaction Value"
    ],
    "value": [
        total_payments,
        int(valid_payments),
        int(invalid_payments),
        round(total_revenue, 2),
        round(average_payment, 2)
    ]
})

payment_kpi.to_csv(
    aggregation_path / "payment_kpis.csv",
    index=False
)


# ============================================================
# F. PAYMENT METHOD KPI
# ============================================================

payment_method_kpi = (
    payments
    .groupby(
        "payment_method",
        dropna=False
    )
    .agg(
        total_transactions=(
            "payment_id",
            "count"
        ),
        valid_transactions=(
            "amount_valid",
            "sum"
        ),
        total_revenue=(
            "amount",
            "sum"
        ),
        average_transaction_value=(
            "amount",
            "mean"
        )
    )
    .reset_index()
)

payment_method_kpi["total_revenue"] = (
    payment_method_kpi["total_revenue"]
    .round(2)
)

payment_method_kpi["average_transaction_value"] = (
    payment_method_kpi["average_transaction_value"]
    .round(2)
)

payment_method_kpi.to_csv(
    aggregation_path / "payment_method_kpis.csv",
    index=False
)


# ============================================================
# G. DATA QUALITY KPI — CORRECTED
# ============================================================

dq_report_path = Path("..") / "output" / "dq_report.csv"

dq_report = pd.read_csv(dq_report_path)

# Make sure failed counts are numeric
dq_report["failed_record_count"] = pd.to_numeric(
    dq_report["failed_record_count"],
    errors="coerce"
).fillna(0)

total_rules = len(dq_report)

rules_with_violations = (
    dq_report["failed_record_count"] > 0
).sum()

total_violation_events = (
    dq_report["failed_record_count"]
    .sum()
)

# Count records currently in quarantine
quarantined_records = 0

for file in quarantine_path.glob("*.csv"):

    qdf = pd.read_csv(file)

    quarantined_records += len(qdf)


# DQ rule pass rate
dq_pass_rate = (
    (total_rules - rules_with_violations)
    / total_rules
    * 100
    if total_rules > 0
    else 0
)


dq_kpi = pd.DataFrame({
    "metric": [
        "Total DQ Rules",
        "Rules With Violations",
        "Total Violation Events",
        "Quarantined Records",
        "DQ Rule Pass Rate (%)"
    ],
    "value": [
        total_rules,
        int(rules_with_violations),
        int(total_violation_events),
        int(quarantined_records),
        round(dq_pass_rate, 2)
    ]
})


dq_kpi.to_csv(
    aggregation_path / "dq_kpis.csv",
    index=False
)


print("=" * 60)
print("DATA QUALITY KPI")
print("=" * 60)

print(
    dq_kpi.to_string(index=False)
)

print("\ndq_kpis.csv created successfully")


# ============================================================
# H. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("KPI GENERATION COMPLETED")
print("=" * 70)

for file in sorted(aggregation_path.glob("*.csv")):

    df = pd.read_csv(file)

    print(
        f"{file.name:<30} {len(df):>5} rows"
    )

print("\nLocation:")
print(aggregation_path)

print("\nFiles created:")
print("""
1. flight_kpis.csv
2. route_kpis.csv
3. airline_kpis.csv
4. booking_kpis.csv
5. payment_kpis.csv
6. payment_method_kpis.csv
7. dq_kpis.csv
""")

print("=" * 70)

ASG AIRLINES — KPI & AGGREGATION LAYER

Aggregation directory: ..\data\gold\aggregations
DATA QUALITY KPI
                metric   value
        Total DQ Rules   23.00
 Rules With Violations   14.00
Total Violation Events 1651.00
   Quarantined Records   79.00
 DQ Rule Pass Rate (%)   39.13

dq_kpis.csv created successfully

KPI GENERATION COMPLETED
airline_kpis.csv                   5 rows
booking_kpis.csv                   8 rows
dq_kpis.csv                        5 rows
flight_kpis.csv                    6 rows
payment_kpis.csv                   5 rows
payment_method_kpis.csv            3 rows
route_kpis.csv                    30 rows

Location:
..\data\gold\aggregations

Files created:

1. flight_kpis.csv
2. route_kpis.csv
3. airline_kpis.csv
4. booking_kpis.csv
5. payment_kpis.csv
6. payment_method_kpis.csv
7. dq_kpis.csv

